# Elderly Health Monitoring AI - Master Pipeline
## (基于项目结构的完整流程)

This notebook acts as the driver for the entire system. It **imports** logic from your organized folder structure:
- `src/data/`: Ingestion and Preprocessing
- `src/models/`: AI Model Architectures
- `src/utils/`: Logging and Helpers

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

# 1. Add 'src' to the Python path so we can import our modules
sys.path.append(os.path.abspath('../src'))

# 2. Import your existing Python files
from data.loaders import DataLoader
from data.preprocessor import DataPreprocessor
from models.health_risk_model import HealthRiskModel
from models.fall_detection_model import FallDetectionModel
from utils.logger import setup_logger

print("Modules imported from src/ successfully.")

### Step 1: Data Ingestion (WESAD & SisFall)
We use the `DataLoader` class from `src/data/loaders.py`.

In [ ]:
loader = DataLoader(base_path='../data/')
preprocessor = DataPreprocessor()

wesad_subjects = ['S2', 'S3'] # Add more subjects as needed
X_all_wesad, y_all_wesad = [], []

print("Ingesting WESAD data...")
for sub in tqdm(wesad_subjects):
    data = loader.load_wesad_subject(sub)
    if data:
        signals, labels = data
        # Use Preprocessor from src/data/preprocessor.py
        norm_acc = preprocessor.normalize_series(signals['ACC'])
        X, y = preprocessor.create_sequences(norm_acc, labels, seq_length=700, step=700)
        X_all_wesad.append(X)
        y_all_wesad.append(y)

X_wesad = np.vstack(X_all_wesad)
y_wesad = np.concatenate(y_all_wesad)
print(f"Total WESAD sequences: {len(X_wesad)}")

### Step 2: Training the Health Risk Model
We use the `HealthRiskModel` class from `src/models/health_risk_model.py`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_wesad, y_wesad, test_size=0.2)

risk_model = HealthRiskModel(input_shape=(700, 3))
print("Starting LSTM Training...")
history = risk_model.train(X_train[:500], y_train[:500], epochs=5, batch_size=32)

# Plotting results directly in the notebook
plt.plot(history.history['accuracy'])
plt.title('WESAD Risk Model Accuracy')
plt.show()

### Step 3: Fall Detection Pipeline
We use the `FallDetectionModel` from `src/models/fall_detection_model.py`.

In [ ]:
print("Loading SisFall SA01 Sample...")
sis_data = loader.load_sisfall_file('SA01', 'D01_SA01_R01.txt')

if sis_data:
    df, label = sis_data
    acc_features = df.iloc[:, :3].values
    
    fall_detector = FallDetectionModel()
    # Dummy training for demonstration
    fall_detector.train(acc_features[:1000], np.zeros(1000))
    print("Fall Detection model initialized and tested.")

### Step 4: Model Export
Saving the trained models to the `models/` directory.

In [ ]:
os.makedirs('../models', exist_ok=True)
risk_model.save('../models/health_risk_final.h5')
print("Final models exported to /models/")